#### Обучите модель линейной регрессии на найденных двумя способами трёх важных признаках и сравните полученные результаты. Загрузите полученный ноутбук (в формате IPYNB) в форму ниже.

##### КРИТЕРИИ ОЦЕНИВАНИЯ:

1 балл	Верно выделены три столбца-признака для обучения, выбранные RFE.

1 балл	Верно выделены три столбца-признака для обучения, выбранные SelectKBest.

3 балла	Обучена регрессия на первых трёх столбцах, оценено качество модели на тесте.

3 балла	Обучена регрессия на вторых трёх столбцах, оценено качество модели на тесте.

2 балла	Произведено сравнение выбранных метрик в форме комментария. Дан ответ на вопрос «Какой метод отбора признаков показал наилучший результат на тестовой выборке?» (в текстовой ячейке).

Максимальное количество баллов за выполнение задания — 10.

Импорт и загрузка

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import RFE, SelectKBest, f_regression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

data = pd.read_excel('data_ford_price.xlsx')
data.head()

,price,year,condition,cylinders,odometer,title_status,transmission,drive,size,lat,long,weather
0,43900,2016,4,6,43500,clean,automatic,4wd,full-size,36.471500,-82.483400,59.0
1,15490,2009,2,8,98131,clean,automatic,4wd,full-size,40.468826,-74.281734,52.0
2,2495,2002,2,8,201803,clean,automatic,4wd,full-size,42.477134,-82.949564,45.0
3,1300,2000,1,8,170305,rebuilt,automatic,4wd,full-size,40.764373,-82.349503,49.0
4,13865,2010,3,8,166062,clean,automatic,4wd,NaN,49.210949,-123.114720,NaN


Подготовка данных

In [2]:
y = data['price']
X = data.drop(columns=['price'])

X = pd.get_dummies(X, drop_first=True)
X = X.fillna(X.median(numeric_only=True))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

RFE

In [3]:
rfe = RFE(estimator=LinearRegression(), n_features_to_select=3)
rfe.fit(X_train, y_train)

rfe_features = X_train.columns[rfe.support_]
print("RFE selected features:", list(rfe_features))

X_train_rfe = X_train[rfe_features]
X_test_rfe = X_test[rfe_features]

model_rfe = LinearRegression()
model_rfe.fit(X_train_rfe, y_train)
pred_rfe = model_rfe.predict(X_test_rfe)

rfe_r2 = r2_score(y_test, pred_rfe)
rfe_mse = mean_squared_error(y_test, pred_rfe)
rfe_mae = mean_absolute_error(y_test, pred_rfe)

print("RFE model quality:")
print("R2:", rfe_r2)
print("MSE:", rfe_mse)
print("MAE:", rfe_mae)

RFE selected features: ['condition', 'drive_fwd', 'size_sub-compact']
RFE model quality:
R2: 0.1548846121398132
MSE: 160861567.58047655
MAE: 8072.499886937252


SelectKBest

In [6]:
skb = SelectKBest(score_func=f_regression, k=3)
skb.fit(X_train, y_train)

skb_features = X_train.columns[skb.get_support()]
print("SelectKBest selected features:", list(skb_features))

X_train_skb = X_train[skb_features]
X_test_skb = X_test[skb_features]

model_skb = LinearRegression()
model_skb.fit(X_train_skb, y_train)
pred_skb = model_skb.predict(X_test_skb)

skb_r2 = r2_score(y_test, pred_skb)
skb_mse = mean_squared_error(y_test, pred_skb)
skb_mae = mean_absolute_error(y_test, pred_skb)

print("SelectKBest model quality:")
print("R2:", skb_r2)
print("MSE:", skb_mse)
print("MAE:", skb_mae)

SelectKBest selected features: ['year', 'condition', 'odometer']
SelectKBest model quality:
R2: 0.4174296911999369
MSE: 110888021.26381831
MAE: 5187.81164748476


Сравнение

In [7]:
if skb_r2 > rfe_r2:
    best_method = "SelectKBest"
else:
    best_method = "RFE"

print("Best method:", best_method)

Best method: SelectKBest


Мы обучили линейную регрессию на двух наборах признаков, отобранных методами RFE и SelectKBest.  
По метрикам на тестовой выборке лучший результат показала модель на признаках, выбранных SelectKBest: у неё выше R2 и ниже MSE/MAE.  
Следовательно, на данной выборке лучший метод отбора признаков — SelectKBest.